<a href="https://colab.research.google.com/github/debanjanhati2-boop/Basic-Chatbot/blob/main/AI_ER_Diagram_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.tokenize import word_tokenize
import re
import networkx as nx
import matplotlib.pyplot as plt
import json
import warnings
warnings.filterwarnings('ignore')

In [8]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')

class ERDiagramGenerator:
    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))
        self.entities = {}
        self.relationships = []
        self.attributes = {}
        self.entity_types = {}

    def preprocess_text(self, text):
        text = text.lower()
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        tokens = word_tokenize(text)
        tokens = [self.lemmatizer.lemmatize(token) for token in tokens
                    if token not in self.stop_words and len(token) > 2]
        return ' '.join(tokens)

    def extract_entities(self, text):
        tokens = word_tokenize(text)
        pos_tags = pos_tag(tokens)

        entities = []
        current_entity = []
        entity_count = 0

        for word, tag in pos_tags:
            if tag.startswith('NN') and word.lower() not in self.stop_words:
                if not current_entity:
                    entity_count += 1
                current_entity.append(word)
            else:
                if current_entity:
                    entity_name = ' '.join(current_entity).title()
                    if len(entity_name) > 2 and entity_name not in entities:
                        entities.append(entity_name)
                    current_entity = []

        if current_entity:
            entity_name = ' '.join(current_entity).title()
            if len(entity_name) > 2 and entity_name not in entities:
                entities.append(entity_name)

        for entity in entities:
            if entity not in self.entities:
                self.entities[entity] = {
                    'attributes': [],
                    'primary_key': None,
                    'type': 'Regular'
                }

        return entities

    def extract_attributes(self, text):


        attribute_patterns = [
            r'(\w+)\s+(?:has|contains|includes)\s+(\w+)',
            r'(\w+)\s+(?:with|having)\s+(\w+)',
            r'attributes?\s+(?:of|for)\s+(\w+)\s+(?:are|include)\s+([\w\s,]+)'
        ]

        for pattern in attribute_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            for match in matches:
                if len(match) >= 2:
                    entity = match[0].title()
                    attr = match[1].title()
                    if entity in self.entities and attr not in self.entities[entity]['attributes']:
                        self.entities[entity]['attributes'].append(attr)

    def extract_relationships(self, text):
        relationship_patterns = [
            r'(\w+)\s+(?:has|contains|includes|manages)\s+(\w+)',
            r'(\w+)\s+(?:belongs to|is part of)\s+(\w+)',
            r'(\w+)\s+(?:works with|interacts with)\s+(\w+)',
            r'(\w+)\s+(?:related to|associated with)\s+(\w+)'
        ]

        relationship_types = {
            'has': 'One-to-Many',
            'contains': 'One-to-Many',
            'belongs to': 'Many-to-One',
            'manages': 'One-to-One',
            'works with': 'Many-to-Many',
            'related to': 'Many-to-Many'
        }

        for pattern in relationship_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            for match in matches:
                if len(match) >= 2:
                    entity1 = match[0].title()
                    entity2 = match[1].title()

                    rel_type = 'Many-to-Many'
                    for word, rtype in relationship_types.items():
                        if word in text.lower():
                            rel_type = rtype
                            break

                    if entity1 in self.entities and entity2 in self.entities:
                        relationship = {
                            'from': entity1,
                            'to': entity2,
                            'type': rel_type,
                            'label': 'Relates to'
                        }
                        if relationship not in self.relationships:
                            self.relationships.append(relationship)

    def generate_er_diagram(self, text):
        self.entities = {}
        self.relationships = []
        self.extract_entities(text)
        self.extract_attributes(text)
        self.extract_relationships(text)

        for entity in self.entities:
            if not self.entities[entity]['attributes']:
                self.entities[entity]['attributes'] = ['id', 'name']
                self.entities[entity]['primary_key'] = 'id'

        return {
            'entities': self.entities,
            'relationships': self.relationships
        }

    def visualize_er_diagram(self, title="Entity-Relationship Diagram"):

        G = nx.DiGraph()


        for entity in self.entities:
            G.add_node(entity,
                       label=f"{entity}\n({', '.join(self.entities[entity]['attributes'])})",
                       color='lightblue')

        for rel in self.relationships:
            G.add_edge(rel['from'], rel['to'],
                       label=f"{rel['type']}")


        fig, ax = plt.subplots(figsize=(12, 8))


        pos = nx.spring_layout(G, k=2, iterations=50)


        node_colors = ['lightblue' for _ in G.nodes()]
        nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                               node_size=3000, ax=ax)

        nx.draw_networkx_edges(G, pos, edge_color='gray',
                               arrows=True, arrowstyle='-|>',
                               arrowsize=20, ax=ax)

        nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold', ax=ax)

        edge_labels = {(rel['from'], rel['to']): rel['type']
                      for rel in self.relationships}
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                                      font_size=8, ax=ax)

        ax.set_title(title, fontsize=16, fontweight='bold')
        ax.axis('off')

        plt.tight_layout()
        return fig

    def generate_sql_schema(self):

        sql = "-- SQL Schema Generated from ER Diagram\n"
        sql += "-- Free Online AI & Data Science Internship Task AI-PR-006\n\n"

        for entity, data in self.entities.items():
            sql += f"CREATE TABLE {entity.lower()} (\n"
            attributes = data['attributes']
            for i, attr in enumerate(attributes):
                is_pk = " PRIMARY KEY" if attr == data['primary_key'] else ""
                comma = "," if i < len(attributes) - 1 else ""
                sql += f"    {attr} VARCHAR(255){is_pk}{comma}\n"
            sql += ");\n\n"


        for rel in self.relationships:
            sql += f"-- Relationship: {rel['from']} to {rel['to']} ({rel['type']})\n"
            sql += f"ALTER TABLE {rel['to'].lower()} ADD COLUMN {rel['from'].lower()}_id INT;\n"
            sql += f"ALTER TABLE {rel['to'].lower()} ADD FOREIGN KEY ({rel['from'].lower()}_id) REFERENCES {rel['from'].lower()}(id);\n\n"

        return sql

    def generate_report(self, text):

        er_data = self.generate_er_diagram(text)

        report = f"""
📊 Entity-Relationship Diagram Report
{'='*50}

📋 Input Text Analysis:
  - Text Length: {len(text)} characters
  - Words: {len(text.split())}

📊 Entities Found:
  - Total Entities: {len(er_data['entities'])}
"""

        for entity, data in er_data['entities'].items():
            report += f"\n  📌 {entity}:\n"
            report += f"     Attributes: {', '.join(data['attributes'])}\n"
            report += f"     Primary Key: {data['primary_key'] or 'Not Set'}\n"
            report += f"     Type: {data['type']}\n"

        report += f"\n🔗 Relationships Found:\n"
        if er_data['relationships']:
            for rel in er_data['relationships']:
                report += f"  - {rel['from']} → {rel['to']} ({rel['type']})\n"
        else:
            report += "  No relationships detected\n"

        report += """
💡 Database Design Recommendations:
  1. Ensure each entity has a primary key
  2. Normalize to at least 3NF
  3. Add foreign key constraints for relationships
  4. Consider indexing on frequently queried columns
  5. Use appropriate data types for attributes

📝 SQL Schema Generated: Ready for export
"""
        return report

# Usage Example
generator = ERDiagramGenerator()

text = """
This is a Student Management System. The Student entity has attributes like student_id,
name, email, and phone. The Course entity has course_id, title, and credits.
Students enroll in courses. The Department entity has dept_id, name, and location.
Each student belongs to one department. Each course is offered by one department.
Professors teach courses and have professor_id, name, and specialization.
Students have a relationship with professors through courses.
"""


print("📊 Generating ER Diagram...")
print("="*50)
er_data = generator.generate_er_diagram(text)

print(f"✅ Extracted {len(er_data['entities'])} entities")
print(f"✅ Found {len(er_data['relationships'])} relationships")

print("\n📌 Entities:")
for entity, data in er_data['entities'].items():
    print(f"  {entity}: {', '.join(data['attributes'])}")

print("\n🔗 Relationships:")
for rel in er_data['relationships']:
    print(f"  {rel['from']} → {rel['to']} ({rel['type']})")

print("\n📝 SQL Schema:")
print("="*50)
print(generator.generate_sql_schema())

print("\n📄 Comprehensive Report:")
print(generator.generate_report(text))

print("\n📊 Visualizing ER Diagram...")
fig = generator.visualize_er_diagram("Student Management System ER Diagram")
fig.savefig('er_diagram.png', dpi=100, bbox_inches='tight')
print("✅ ER diagram saved as 'er_diagram.png'")
plt.close()

print("\n🧪 Testing Different Domains:")
print("="*50)
test_texts = [
    "E-Commerce system has Products, Customers, Orders, and Payments. Products belong to Categories.",
    "Hospital system includes Patients, Doctors, Appointments, and Departments. Doctors have specializations.",
    "Library system has Books, Members, Borrowing records, and Publishers. Books are written by Authors."
]

for i, test_text in enumerate(test_texts, 1):
    generator.generate_er_diagram(test_text)
    print(f"\nTest {i}:")
    print(f"  Entities: {list(generator.entities.keys())}")
    print(f"  Relationships: {len(generator.relationships)}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


📊 Generating ER Diagram...
✅ Extracted 20 entities
✅ Found 0 relationships

📌 Entities:
  Student Management System: id, name
  Student Entity: id, name
  Attributes: id, name
  Student_Id: id, name
  Name: id, name
  Email: id, name
  Phone: id, name
  Course Entity: id, name
  Title: id, name
  Credits: id, name
  Students: id, name
  Courses: id, name
  Department Entity: id, name
  Location: id, name
  Student: id, name
  Department: id, name
  Course: id, name
  Professors: id, name
  Specialization: id, name
  Relationship: Professors

🔗 Relationships:

📝 SQL Schema:
-- SQL Schema Generated from ER Diagram
-- Free Online AI & Data Science Internship Task AI-PR-006

CREATE TABLE student management system (
    id VARCHAR(255) PRIMARY KEY,
    name VARCHAR(255)
);

CREATE TABLE student entity (
    id VARCHAR(255) PRIMARY KEY,
    name VARCHAR(255)
);

CREATE TABLE attributes (
    id VARCHAR(255) PRIMARY KEY,
    name VARCHAR(255)
);

CREATE TABLE student_id (
    id VARCHAR(255) 